In [1]:
import sqlite3

conn = sqlite3.connect("chinook.db")

def get_full_schema() -> str:
    schema = ""
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master")
    for r in cursor.fetchall():
        schema += (r[0] + "\r\n") if r[0] is not None else ""
    cursor.close()
    return schema

print(get_full_schema())

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)
CREATE TABLE sqlite_sequence(name,seq)
CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
)
CREATE TABLE "customers"
(
    [CustomerId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVARCHAR(60)  NOT NULL,
    [SupportRepId] INTEGER,
    FOREIGN KEY ([SupportRepId]) REFERENCES "employees" ([EmployeeId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)
CREATE TABLE "employees"
(

In [2]:
from transformers import MistralCommonBackend, Mistral3ForConditionalGeneration, FineGrainedFP8Config

model_id = "mistralai/Ministral-3-14B-Instruct-2512"
tokenizer = MistralCommonBackend.from_pretrained(model_id)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=FineGrainedFP8Config(dequantize=True)
)

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


tekken.json:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/585 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

In [9]:
import re
import torch

def text_to_sql(query: str) -> str:
    prompt = """Tu es un expert SQL qui doit écrire des requêtes sur le schéma suivant, à partir de la demande utilisateur. Tu ne dois rien écrire d'autre que la requête SQL.
Schéma : {schema}
Demande utilisateur : {query}
Réponse :
""".format(
        schema=get_full_schema(),
        query=query
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    with torch.no_grad():
        generation_output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    output_text = tokenizer.decode(generation_output[0], skip_special_tokens=True).strip()

    match = re.search(r"Réponse\s*:\s*(.*)", output_text, flags=re.DOTALL)
    if match:
        sql_query = match.group(1).strip()
    else:
        sql_query = output_text

    sql_query = sql_query.replace("\\n", " ")
    sql_query = re.sub(r"\\", "", sql_query)
    sql_query = re.sub(r"^```sql\s*|^```\s*|\s*```$", "", sql_query, flags=re.IGNORECASE).strip()

    return sql_query

In [10]:
#Exécution de la requête SQL
def execute_sql(query: str) -> str:
    result = []
    cursor = conn.cursor()
    cursor.execute(query)
    for r in cursor.fetchall():
        result.append(r)
    cursor.close()
    return result

execute_sql("SELECT COUNT(*) FROM albums;")

[(347,)]

In [11]:
import re
import torch

def format_sql_response(query: str, result: str) -> str:
    prompt = f"""Ton rôle est d'écrire une réponse à une question dans un format interprétable pour un humain, dans la même langue que la demande d'origine.
Tu as également le résultat qui provient de la base de données que tu peux utiliser pour formuler la réponse à l'utilisateur.
Tu dois être clair, concis, formel sans ajouter d'émotion ou de détails dont tu n'as pas connaissance dans le résultat ou la demande d'origine.
Demande d'origine : {query}
Résultat base de données : {result}
Réponse : """
    
    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    with torch.no_grad():
        generation_output = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False
        )

    output_text = tokenizer.decode(generation_output[0], skip_special_tokens=True).strip()

    match = re.search(r"Réponse\s*:\s*(.*)", output_text, flags=re.DOTALL)
    if match:
        response = match.group(1).strip()
    else:
        response = output_text

    response = response.replace("\\", "")
    return response.strip()

In [12]:
#Enchaînement
def run_query(query: str) -> str:
    sql_query = text_to_sql(query)
    sql_query = re.sub(r"^```sql\s*|^```\s*|\s*```$", "", sql_query, flags=re.IGNORECASE).strip()
    try:
        print("Exécution de la requête suivante :", sql_query)
        sql_result = execute_sql(sql_query)
    except Exception:
        print("Erreur de requête SQL :", sql_query)
        return
    return format_sql_response(query, str(sql_result))

run_query("Combien y a-t-il d'albums ?")

Exécution de la requête suivante : SELECT COUNT(*) AS nombre_albums FROM albums;


'347 albums sont disponibles.'